## Flow Cytometry Analysis — Extra Phenotypes (PARP/yH2AX Cell Cycle)
### Author: Eleni Aretaki
### Date: 2026-03-25
### Purpose: 
This notebook analyzes the cell cycle distribution of DNA damage–positive populations
(PARP-positive and γH2AX-positive cells) across SWI/SNF knockout cell lines.

The analysis is performed using data from both 48h flow cytometry experiments.
### Analysis Overview:
1. Load and merge PARP and γH2AX cell cycle datasets from both experiments
2. Annotate samples with cell line and treatment metadata
3. Remove excluded wells and inactive treatments
4. Generate stacked barplots comparing:
DNA damage positive vs negative populations (DMSO only)
Drug-treated vs DMSO within DNA damage positive populations
### Outputs:
The notebook generates two figure types:

1. Baseline genotype effect (DMSO only)
    - PARP− vs PARP+ cell cycle distribution
    - γH2AX− vs γH2AX+ cell cycle distribution
    
2. Treatment-induced shifts (positive populations only)
    - DMSO vs drug comparison for PARP+ cells
    - DMSO vs drug comparison for γH2AX+ cells

#### Cell Lines: ARID1A, ARID1B, ARID2, SMARCA4, BRD9, PBRM1, SMARCC1, SMARCD1, BRD7, PHF10, WT, WT+BRM014
#### Compounds: Camptothecin, MMS, Adavosertib, Palbociclib, MLN4924, Hydroxyurea, BIBR1523, Cobimetinib, Lapatinib, Niraparib, Paclitaxel

In [ ]:
# -------------------------------
# Imports
# -------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import glob
import os

In [ ]:
# -------------------------------
# Loading data
# -------------------------------
def load_marker_data(folder, marker_keyword, annotation_df, experiment_name):
    """
    Loads all replicate CSVs for a given marker (e.g. PARP, yH2AX)
    from a folder and returns annotated dataframe.
    """

    # find all relevant files
    pattern = os.path.join(folder, f"*{marker_keyword}*cellcycle.csv")
    files = glob.glob(pattern)

    dfs = []

    for file in files:
        df = pd.read_csv(file)

        # Extract replicate number from filename
        rep = os.path.basename(file).split("_")[1]  # Replicate_X
        df["Replicate"] = rep

        # Identify positive vs negative
        df["Population"] = "neg" if "neg" in file else "pos"

        # Add experiment label
        df["Experiment"] = experiment_name

        dfs.append(df)

    combined = pd.concat(dfs, ignore_index=True)

    # Extract well ID
    combined["well_ID"] = combined["Sample"].str.extract(r"_([A-H]\d{1,2})_")

    # Merge annotation
    combined = combined.merge(annotation_df, on="well_ID", how="left")

    return combined

In [ ]:
annotation1 = pd.read_csv("../annotation_df.csv", encoding='unicode_escape')
annotation2 = pd.read_csv("../annotation_df2.csv", encoding='unicode_escape')

# -------------------------------
# Loading PARP data
# -------------------------------
# Experiment 1
exp1 = load_marker_data(
    folder="../Exp1",
    marker_keyword="PARP",
    annotation_df=annotation1,
    experiment_name="Exp1"
)

# Experiment 2
exp2 = load_marker_data(
    folder="../Exp2",
    marker_keyword="PARP",
    annotation_df=annotation2,
    experiment_name="Exp2"
)

# Combine everything
all_data = pd.concat([exp1, exp2], ignore_index=True)

# -------------------------------
# Loading yH2AX data
# -------------------------------
# Experiment 1
y_exp1 = load_marker_data(
    folder="../Exp1",
    marker_keyword="yH2AX",
    annotation_df=annotation1,
    experiment_name="Exp1"
)

# Experiment 2
y_exp2 = load_marker_data(
    folder="../Exp2",
    marker_keyword="yH2AX",
    annotation_df=annotation2,
    experiment_name="Exp2"
)

# Combine everything
y_all_data = pd.concat([y_exp1, y_exp2], ignore_index=True)

In [ ]:
# -------------------------------
# Removing outliers
# -------------------------------
def remove_bad_wells(df, bad_dict):
    """
    bad_dict format:
    {
        "Exp1": ["A1","B2"],
        "Exp2": ["C3"]
    }
    """
    df_clean = df.copy()

    for exp, rep_dict in bad_dict.items():
        for rep, wells in rep_dict.items():
            
            df_clean = df_clean[
                ~(
                    (df_clean["Experiment"] == exp) &
                    (df_clean["Replicate"] == rep) &
                    (df_clean["well_ID"].isin(wells))
                )
            ]

    return df_clean

bad_wells = {
    "Exp1": {
        "Rep1": ["E8"],
        "Rep2": ["B3", "B4"],
        "Rep3": ["E12", "F12"]
    },
    "Exp2": {
        "Rep1": ["A8"],
        "Rep2": ["A8"],
        "Rep3": ["A8"],
        "Rep5": ["A8", "B7", "C12"]
    }
}

clean_data = remove_bad_wells(all_data, bad_wells)
clean_data_y = remove_bad_wells(y_all_data, bad_wells)

bad_treatments = ["Aph", "NIR"]

clean_data = clean_data[
    ~clean_data["treatment"].isin(bad_treatments)
].copy()

clean_data_y = clean_data_y[
    ~clean_data_y["treatment"].isin(bad_treatments)
].copy()

In [ ]:
# -------------------------------
# Plotting Functions
# -------------------------------
def plot_all_celllines_stacked(pos_df, neg_df, marker_name,
                                treatment_name="DMSO",
                                output_dir="stacked_plots"):
    """
    Automatically generates stacked bar plots (mean ± SEM)
    for all cell lines under a given treatment
    and saves them as separate image files.
    """

    # Create output folder if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    plt.rcParams['font.family'] = 'Arial'
    
    # Filter for treatment
    pos_filtered = pos_df[pos_df["treatment"] == treatment_name]
    neg_filtered = neg_df[neg_df["treatment"] == treatment_name]

    # Find common cell lines present in both dfs
    cell_lines = sorted(
        set(pos_filtered["modification"]).intersection(
            set(neg_filtered["modification"])
        )
    )

    for cell_line in cell_lines:

        pos = pos_filtered[pos_filtered["modification"] == cell_line]
        neg = neg_filtered[neg_filtered["modification"] == cell_line]

        if pos.empty or neg.empty:
            continue

        # Calculate mean and SEM
        pos_mean = pos[["G1", "S", "G2"]].mean()
        neg_mean = neg[["G1", "S", "G2"]].mean()

        pos_sem = pos[["G1", "S", "G2"]].sem()
        neg_sem = neg[["G1", "S", "G2"]].sem()

        labels = [f"{marker_name}−", f"{marker_name}+"]
        x = range(2)

        G1_vals = [neg_mean["G1"], pos_mean["G1"]]
        S_vals = [neg_mean["S"], pos_mean["S"]]
        G2_vals = [neg_mean["G2"], pos_mean["G2"]]

        G1_err = [neg_sem["G1"], pos_sem["G1"]]
        S_err = [neg_sem["S"], pos_sem["S"]]
        G2_err = [neg_sem["G2"], pos_sem["G2"]]

        plt.figure(figsize=(2.75, 4))

        # G1 (bottom)
        plt.bar(
            x, 
            G1_vals, 
            yerr=G1_err,
            color='steelblue',
            width=0.5,
            capsize=1.5, 
            label="G1", 
            error_kw={'elinewidth': 0.6, 'capthick': 0.6})

        # S (middle)
        plt.bar(
            x,
            S_vals,
            bottom=G1_vals,
            yerr=S_err,
            color='lightblue',
            width=0.5,
            capsize=1.5,
            error_kw={'elinewidth': 0.6, 'capthick': 0.6},
            label="S"
        )

        # G2 (top)
        bottom_G2 = [i + j for i, j in zip(G1_vals, S_vals)]
        plt.bar(
            x,
            G2_vals,
            bottom=bottom_G2,
            yerr=G2_err,
            color='thistle',
            width=0.5,
            capsize=1.5,
            error_kw={'elinewidth': 0.6, 'capthick': 0.6},
            label="G2/M"
        )


        cell_line_label = (
                cell_line.replace("C631 + BRM014", "WT+BRM014")
               .replace("C631", "WT")
               .replace(" KO", "")
        )
        
        plt.xticks(x, labels)
        plt.ylabel("Cells (%)")
        plt.title(f"{cell_line_label} – {treatment_name}")
        plt.ylim(0, 103)
        plt.legend()

        plt.tight_layout()

        # Save figure
        filename = f"{cell_line_label}_{treatment_name}.pdf"
        plt.savefig(os.path.join(output_dir, filename), dpi=600)

        plt.close()  

    print(f"Plots saved in folder: {output_dir}")



def plot_positive_treatment_vs_dmso(pos_df,
                                    marker_name="PARP",
                                    output_dir="treatment_vs_dmso"):

    """
    For positive populations only (PARP+ or γH2AX+),
    compares each treatment to DMSO within each cell line.

    Generates stacked bar plots (mean ± SEM) and saves them.
    NOTE: We do **NOT** combine data from the two experiments, 
    as we use the same cell lines in both experiments and 
    we do not want to mix the DMSO samples for more accurate/correct comparisons with the treatments.
    """

    os.makedirs(output_dir, exist_ok=True)
    plt.rcParams['font.family'] = 'Arial'

    # Get all cell lines
    cell_lines = sorted(pos_df["modification"].unique())

    for cell_line in cell_lines:

        df_cl = pos_df[pos_df["modification"] == cell_line]

        # Identify treatments for this cell line
        treatments = sorted(df_cl["treatment"].unique())

        if "DMSO" not in treatments:
            continue

        treatments.remove("DMSO")

        for treatment in treatments:

            df_dmso = df_cl[df_cl["treatment"] == "DMSO"]
            df_treat = df_cl[df_cl["treatment"] == treatment]

            if df_dmso.empty or df_treat.empty:
                continue

            # Compute mean and SEM
            dmso_mean = df_dmso[["G1", "S", "G2"]].mean()
            treat_mean = df_treat[["G1", "S", "G2"]].mean()

            dmso_sem = df_dmso[["G1", "S", "G2"]].sem()
            treat_sem = df_treat[["G1", "S", "G2"]].sem()

            labels = ["DMSO", treatment]
            x = range(2)

            G1_vals = [dmso_mean["G1"], treat_mean["G1"]]
            S_vals = [dmso_mean["S"], treat_mean["S"]]
            G2_vals = [dmso_mean["G2"], treat_mean["G2"]]

            G1_err = [dmso_sem["G1"], treat_sem["G1"]]
            S_err = [dmso_sem["S"], treat_sem["S"]]
            G2_err = [dmso_sem["G2"], treat_sem["G2"]]

            plt.figure(figsize=(2.75, 4))

            # G1
            plt.bar(
                x, 
                G1_vals, 
                yerr=G1_err, 
                color='steelblue',
                width=0.5,
                capsize=1.5,
                error_kw={'elinewidth': 0.6, 'capthick': 0.6}, 
                label="G1")

            # S
            plt.bar(
                x,
                S_vals,
                bottom=G1_vals,
                yerr=S_err,
                color='lightblue',
                width=0.5,
                capsize=1.5,
                error_kw={'elinewidth': 0.6, 'capthick': 0.6},
                label="S"
            )

            # G2/M
            bottom_G2 = [i + j for i, j in zip(G1_vals, S_vals)]
            plt.bar(
                x,
                G2_vals,
                bottom=bottom_G2,
                yerr=G2_err,
                color='thistle',
                width=0.5,
                capsize=1.5,
                error_kw={'elinewidth': 0.6, 'capthick': 0.6},
                label="G2/M"
            )


            cell_line_label = (
                cell_line.replace("C631 + BRM014", "WT+BRM014")
               .replace("C631", "WT")
               .replace(" KO", "")
            )
            
            plt.xticks(x, labels)
            plt.ylabel(f"{marker_name}+ cells (%)")
            plt.title(f"{cell_line_label}")
            plt.ylim(0, 103)
            plt.legend()

            plt.tight_layout()

            save_dir = os.path.join(output_dir, cell_line_label)
            os.makedirs(save_dir, exist_ok=True)
            
            filename = f"{cell_line_label}_{treatment}.pdf"
            plt.savefig(os.path.join(save_dir, filename), dpi=600)

            plt.close()

    print(f"Plots saved in folder: {output_dir}")

In [ ]:
out_dir = "../Extra_Phenotypes"

dmso_df = clean_data[clean_data["treatment"] == "DMSO"]
pos_dmso = dmso_df[dmso_df["Population"] == "pos"]
neg_dmso = dmso_df[dmso_df["Population"] == "neg"]
plot_all_celllines_stacked(pos_dmso, neg_dmso, marker_name = 'PARP', output_dir= os.path.join(out_dir, f"Untreated/PARP-Positive/PDFs"))

dmso_df_y = clean_data_y[clean_data_y["treatment"] == "DMSO"]
pos_dmso_y = dmso_df_y[dmso_df_y["Population"] == "pos"]
neg_dmso_y = dmso_df_y[dmso_df_y["Population"] == "neg"]
plot_all_celllines_stacked(pos_dmso_y, neg_dmso_y, marker_name = 'yH2AX', output_dir= os.path.join(out_dir, f"Untreated/yH2AX-Positive/PDFs"))

In [ ]:
experiments = clean_data["Experiment"].unique()

for exp in experiments:

    df_exp = clean_data[clean_data["Experiment"] == exp]

    plot_positive_treatment_vs_dmso(
        df_exp[df_exp["Population"] == "pos"],
        marker_name="PARP",
        output_dir=os.path.join(out_dir, "Treated", "PARP-Positive", "PDFs", f"{exp}")
    )

In [ ]:
experiments_y = clean_data_y["Experiment"].unique()

for exp in experiments_y:

    df_exp_y = clean_data_y[clean_data_y["Experiment"] == exp]

    plot_positive_treatment_vs_dmso(
        df_exp_y[df_exp_y["Population"] == "pos"],
        marker_name="yH2AX",
        output_dir=os.path.join(out_dir, "Treated", "yH2AX-Positive", "PDFs", f"{exp}")
    )